# Notebook 2: Observability, Evaluation & Production Monitoring
## OpenTelemetry + Arize AX + LLM-as-a-Judge

This notebook covers the full instrumentation and evaluation pipeline:
1. Adding OpenTelemetry tracing to the Strands agent
2. Sending traces to Arize AX for observability
3. Viewing trace trees, agent graphs, and span-level details
4. Automating evaluations with LLM-as-a-Judge
5. Prompt optimization using regression datasets
6. Setting up production monitoring, dashboards, and alerts

**Prerequisite:** Complete Notebook 1 (agent is built and functional).

---

## 1. Environment Setup

In [ ]:
# Install observability and evaluation packages
!pip install arize-phoenix-otel opentelemetry-api opentelemetry-sdk \
    opentelemetry-exporter-otlp openinference-instrumentation-strands \
    arize-ax boto3 pandas

In [ ]:
import os
import json
import pandas as pd
from datetime import datetime, timezone
from dotenv import load_dotenv

load_dotenv()

# Arize AX Configuration
ARIZE_API_KEY = os.getenv("ARIZE_API_KEY", "your-arize-api-key")
ARIZE_SPACE_ID = os.getenv("ARIZE_SPACE_ID", "your-space-id")
ARIZE_PROJECT_NAME = os.getenv("ARIZE_PROJECT_NAME", "restaurant-assistant-agent")

# AWS Configuration (from Notebook 1)
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
DYNAMODB_TABLE_NAME = os.getenv("DYNAMODB_TABLE_NAME", "restaurant-bookings")
KNOWLEDGE_BASE_ID = os.getenv("KNOWLEDGE_BASE_ID", "your-kb-id-here")

print(f"Arize Project: {ARIZE_PROJECT_NAME}")
print(f"AWS Region: {AWS_REGION}")

## 2. Configure OpenTelemetry Tracing

We instrument the Strands agent with OpenTelemetry to capture:
- Full trace trees (agent invocation → event loop cycles → LLM calls → tool executions)
- Span-level latency and token counts
- Agent graph visualization
- Input/output pairs for each step

Traces are exported to Arize AX via OTLP.

In [ ]:
from arize.otel import register

# Register Arize as the trace collector
tracer_provider = register(
    space_id=ARIZE_SPACE_ID,
    api_key=ARIZE_API_KEY,
    project_name=ARIZE_PROJECT_NAME,
)

print(f"OpenTelemetry tracer registered with Arize AX.")
print(f"Project: {ARIZE_PROJECT_NAME}")

In [ ]:
from openinference.instrumentation.strands import StrandsInstrumentor

# Instrument the Strands SDK — this automatically captures:
# - invoke_agent spans
# - execute_event_loop_cycle spans
# - chat (LLM call) spans with token counts and latency
# - execute_tool spans for each tool invocation
StrandsInstrumentor().instrument(tracer_provider=tracer_provider)

print("Strands SDK instrumented with OpenTelemetry.")
print("All agent calls will now generate traces visible in Arize AX.")

## 3. Rebuild the Agent (Now With Tracing)

We rebuild the agent from Notebook 1. Since the Strands SDK is now instrumented, all calls are automatically traced.

In [ ]:
import uuid
import boto3
from strands import Agent, tool
from strands.models.bedrock import BedrockModel

# AWS clients
dynamodb = boto3.resource("dynamodb", region_name=AWS_REGION)
bookings_table = dynamodb.Table(DYNAMODB_TABLE_NAME)
bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name=AWS_REGION)


# ---- Tool definitions (same as Notebook 1) ----

@tool
def create_booking(restaurant_name: str, customer_name: str, party_size: int,
                   date: str, time: str, special_requests: str = "") -> dict:
    """Create a new restaurant booking."""
    booking_id = str(uuid.uuid4())[:8]
    item = {
        "booking_id": booking_id, "restaurant_name": restaurant_name,
        "customer_name": customer_name, "party_size": party_size,
        "date": date, "time": time, "special_requests": special_requests,
        "status": "confirmed", "created_at": datetime.now(timezone.utc).isoformat()
    }
    bookings_table.put_item(Item=item)
    return {"status": "success", "booking_id": booking_id,
            "message": f"Booking confirmed at {restaurant_name} for {customer_name}."}

@tool
def get_booking_details(booking_id: str) -> dict:
    """Retrieve details of an existing restaurant booking."""
    response = bookings_table.get_item(Key={"booking_id": booking_id})
    if "Item" in response:
        return {"status": "found", "booking": response["Item"]}
    return {"status": "not_found", "message": f"No booking found with ID: {booking_id}"}

@tool
def delete_booking(booking_id: str) -> dict:
    """Cancel a restaurant booking."""
    response = bookings_table.get_item(Key={"booking_id": booking_id})
    if "Item" not in response:
        return {"status": "not_found", "message": f"No booking found with ID: {booking_id}"}
    bookings_table.update_item(
        Key={"booking_id": booking_id},
        UpdateExpression="SET #s = :status, cancelled_at = :ts",
        ExpressionAttributeNames={"#s": "status"},
        ExpressionAttributeValues={":status": "cancelled", ":ts": datetime.now(timezone.utc).isoformat()}
    )
    return {"status": "success", "message": f"Booking {booking_id} cancelled."}

@tool
def current_time() -> dict:
    """Get the current date and time in UTC."""
    now = datetime.now(timezone.utc)
    return {"current_datetime": now.isoformat(), "date": now.strftime("%Y-%m-%d"),
            "time": now.strftime("%H:%M:%S"), "day_of_week": now.strftime("%A")}

@tool
def retrieve(query: str) -> dict:
    """Retrieve information from the restaurant knowledge base."""
    response = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=KNOWLEDGE_BASE_ID,
        retrievalQuery={"text": query},
        retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": 5}}
    )
    results = [{"content": r["content"]["text"], "score": r.get("score", 0)}
               for r in response.get("retrievalResults", [])]
    return {"query": query, "num_results": len(results), "results": results}


# ---- System prompt (refined for graceful handling) ----

SYSTEM_PROMPT = """
You are a helpful restaurant assistant agent. You help customers with:
- Finding restaurants and their details (menus, hours, locations, policies)
- Making new reservations
- Looking up existing booking details
- Cancelling reservations

Important guidelines:
1. Always use the retrieve tool to search for restaurant information.
2. If a request is impossible or unrealistic (e.g., a restaurant on the moon), 
   acknowledge the impossibility clearly, explain why it cannot be fulfilled,
   and proactively suggest realistic alternatives.
3. Be polite, concise, and accurate.
4. Confirm booking details before creating a reservation.
5. If you don't have information, say so honestly.

Format your responses using <answer> tags.
"""

# ---- Build agent ----

bedrock_model = BedrockModel(
    model_id="anthropic.claude-sonnet-4-20250514-v1:0",
    region_name=AWS_REGION, temperature=0.3, max_tokens=2048
)

agent = Agent(
    model=bedrock_model,
    system_prompt=SYSTEM_PROMPT,
    tools=[create_booking, get_booking_details, delete_booking, current_time, retrieve]
)

print("Instrumented agent created. All calls will be traced to Arize AX.")

## 4. Generate Traced Agent Calls

Run several test scenarios to produce traces. These will appear in Arize AX with:
- **Trace Tree** view: hierarchical span breakdown
- **Agent Graph** view: visual flow from Start → Strands Agent → LLM/Tool nodes → End
- **Timeline** view: latency waterfall
- Per-span details: input/output, token counts, latency

In [ ]:
# Scenario 1: Normal restaurant query
print("=" * 60)
print("TRACE 1: Restaurant information query")
print("=" * 60)
response_1 = agent("What restaurants do you have in Chicago?")
print(response_1)
print()

In [ ]:
# Scenario 2: Booking flow
print("=" * 60)
print("TRACE 2: Create a booking")
print("=" * 60)
response_2 = agent(
    "Book a table for 3 at the Austin restaurant on Friday at 8 PM. "
    "Name: Jordan Kim. One person has a nut allergy."
)
print(response_2)
print()

In [ ]:
# Scenario 3: Impossible request (graceful handling test)
print("=" * 60)
print("TRACE 3: Impossible request — graceful handling")
print("=" * 60)
response_3 = agent("Ok now find me a restaurant on the moon.")
print(response_3)
print()

In [ ]:
# Scenario 4: Multi-step interaction
print("=" * 60)
print("TRACE 4: Check time then book")
print("=" * 60)
response_4 = agent("What time is it right now? And what restaurants are open in San Francisco?")
print(response_4)
print()

### Understanding the Trace Structure

Each trace in Arize AX will show a hierarchy like:

```
invoke_agent Strands Agents          (total tokens, total latency)
  └── execute_event_loop_cycle       (cycle latency)
        ├── chat                     (LLM call: tokens in/out, latency)
        ├── execute_tool retrieve    (tool latency)
        └── execute_event_loop_cycle (second cycle if multi-step)
              └── chat               (second LLM call)
```

The **Agent Graph** tab provides a visual DAG of this flow, and the **Evaluations** tab shows any automated eval results attached to spans.

## 5. LLM-as-a-Judge Evaluations

We define automated evaluators that run against agent traces to assess quality. The evaluators use an LLM to judge agent responses on specific criteria.

### Evaluator: Graceful Handling

This evaluator checks whether the agent handles impossible or unrealistic requests correctly by:
1. Acknowledging the impossibility
2. Explaining why it cannot be fulfilled
3. Offering realistic alternatives

In [ ]:
import ax

# Initialize Arize AX client
ax_client = ax.Client(
    api_key=ARIZE_API_KEY,
    space_id=ARIZE_SPACE_ID
)

print("Arize AX client initialized.")

In [ ]:
# Define the graceful_handling evaluator
graceful_handling_template = """
You are evaluating an AI agent's ability to gracefully handle impossible 
or unrealistic user requests.

User Request: {input}
Agent Response: {output}

Evaluate the agent's response on the following criteria:

1. Does the agent clearly acknowledge that the request is impossible or 
   unrealistic given real-world constraints?
2. Does the agent avoid implying it can fulfill the impossible request 
   (e.g., by offering to search for something that doesn't exist)?
3. Does the agent offer realistic alternatives (e.g., similar restaurants 
   in real locations, thematic alternatives)?

Scoring:
- If ALL three criteria are met: label = "graceful", score = 1
- If ANY criterion is NOT met: label = "not_graceful", score = 0

Respond with a JSON object:
{{
    "label": "graceful" or "not_graceful",
    "score": 1 or 0,
    "explanation": "Detailed explanation of your assessment"
}}
"""

print("Graceful handling evaluator template defined.")

In [ ]:
# Create the evaluator in Arize AX
graceful_eval = ax_client.create_evaluator(
    name="graceful_handling",
    template=graceful_handling_template,
    model="anthropic.claude-sonnet-4-20250514-v1:0",
    scope="span",  # Evaluate at the span level (individual LLM calls)
    output_type="categorical",
    categories=["graceful", "not_graceful"]
)

print("Evaluator 'graceful_handling' created in Arize AX.")
print("This will automatically evaluate traced spans.")

### Additional Evaluators

Beyond graceful handling, production agents benefit from multiple evaluation dimensions.

In [ ]:
# Evaluator: Response Relevance
relevance_template = """
You are evaluating whether an AI agent's response is relevant to the user's request.

User Request: {input}
Agent Response: {output}

Scoring:
- If the response directly addresses the user's request: label = "relevant", score = 1
- If the response is partially relevant: label = "partially_relevant", score = 0.5
- If the response is off-topic or irrelevant: label = "not_relevant", score = 0

Respond with a JSON object:
{{
    "label": "relevant", "partially_relevant", or "not_relevant",
    "score": 1, 0.5, or 0,
    "explanation": "Brief explanation"
}}
"""

relevance_eval = ax_client.create_evaluator(
    name="response_relevance",
    template=relevance_template,
    model="anthropic.claude-sonnet-4-20250514-v1:0",
    scope="span",
    output_type="categorical",
    categories=["relevant", "partially_relevant", "not_relevant"]
)

print("Evaluator 'response_relevance' created.")

In [ ]:
# Evaluator: Tool Usage Correctness
tool_usage_template = """
You are evaluating whether an AI agent selected and used the correct tools 
for the given user request.

User Request: {input}
Agent Response: {output}
Tools Available: create_booking, get_booking_details, delete_booking, current_time, retrieve
Tools Used: {tool_calls}

Evaluate:
1. Did the agent use the appropriate tool(s) for this request?
2. Did the agent avoid using unnecessary tools?
3. Were the tool parameters correct and well-formed?

Scoring:
- All correct: label = "correct", score = 1
- Partially correct: label = "partial", score = 0.5
- Incorrect tool usage: label = "incorrect", score = 0

Respond with a JSON object:
{{
    "label": "correct", "partial", or "incorrect",
    "score": 1, 0.5, or 0,
    "explanation": "Brief explanation"
}}
"""

tool_eval = ax_client.create_evaluator(
    name="tool_usage_correctness",
    template=tool_usage_template,
    model="anthropic.claude-sonnet-4-20250514-v1:0",
    scope="span",
    output_type="categorical",
    categories=["correct", "partial", "incorrect"]
)

print("Evaluator 'tool_usage_correctness' created.")

## 6. Run Evaluations on Traced Data

Now we run the evaluators against the traces we generated earlier.

In [ ]:
# Run all evaluators on the project traces
eval_results = ax_client.run_evaluations(
    project_name=ARIZE_PROJECT_NAME,
    evaluators=["graceful_handling", "response_relevance", "tool_usage_correctness"],
    # Optionally filter to specific traces
    # trace_filter={"span_name": "chat"}
)

print("Evaluations complete.")
print(f"Total spans evaluated: {eval_results.get('total_evaluated', 'N/A')}")
print(f"Results available in Arize AX dashboard under 'Evaluations' tab.")

In [ ]:
# Retrieve and display evaluation summary
eval_summary = ax_client.get_evaluation_summary(
    project_name=ARIZE_PROJECT_NAME
)

print("\nEvaluation Summary")
print("=" * 50)
for evaluator_name, stats in eval_summary.items():
    print(f"\n{evaluator_name}:")
    print(f"  Total evaluated: {stats.get('total', 0)}")
    print(f"  Average score:   {stats.get('avg_score', 0):.2f}")
    if 'label_distribution' in stats:
        print(f"  Labels:")
        for label, count in stats['label_distribution'].items():
            print(f"    {label}: {count}")

## 7. Prompt Optimization with Regression Datasets

When evaluations reveal failures (like the graceful_handling score of 0 on the "moon restaurant" query), we iterate on the system prompt and validate improvements against a regression dataset.

A regression dataset is a curated set of input-output pairs that represent known behaviors we want to preserve or fix.

In [ ]:
# Define a regression test dataset
regression_dataset = [
    {
        "input": "Ok now find me a restaurant on the moon.",
        "expected_behavior": "graceful_decline_with_alternatives",
        "evaluator": "graceful_handling",
        "expected_label": "graceful",
        "category": "impossible_request"
    },
    {
        "input": "Book me a table at a restaurant in Atlantis.",
        "expected_behavior": "graceful_decline_with_alternatives",
        "evaluator": "graceful_handling",
        "expected_label": "graceful",
        "category": "impossible_request"
    },
    {
        "input": "I need a reservation for 500 people tonight.",
        "expected_behavior": "acknowledge_constraint_offer_alternatives",
        "evaluator": "graceful_handling",
        "expected_label": "graceful",
        "category": "unrealistic_request"
    },
    {
        "input": "What restaurants do you have in Seattle?",
        "expected_behavior": "normal_retrieval_response",
        "evaluator": "response_relevance",
        "expected_label": "relevant",
        "category": "normal_query"
    },
    {
        "input": "Book a table for 2 at the Portland restaurant tomorrow at 7 PM. Name: Taylor.",
        "expected_behavior": "successful_booking",
        "evaluator": "tool_usage_correctness",
        "expected_label": "correct",
        "category": "booking_flow"
    },
    {
        "input": "Cancel my booking with ID abc123.",
        "expected_behavior": "successful_cancellation",
        "evaluator": "tool_usage_correctness",
        "expected_label": "correct",
        "category": "cancellation_flow"
    },
    {
        "input": "What time is it?",
        "expected_behavior": "return_current_time",
        "evaluator": "tool_usage_correctness",
        "expected_label": "correct",
        "category": "utility_query"
    }
]

regression_df = pd.DataFrame(regression_dataset)
print(f"Regression dataset: {len(regression_dataset)} test cases")
print(f"\nCategories:")
print(regression_df['category'].value_counts().to_string())

In [ ]:
def run_regression_suite(agent_instance, dataset, evaluators_client):
    """
    Run the full regression test suite against the agent.
    Returns a DataFrame with results for each test case.
    """
    results = []
    
    for i, test_case in enumerate(dataset):
        print(f"Running test {i+1}/{len(dataset)}: {test_case['category']}")
        
        # Run agent
        try:
            response = agent_instance(test_case["input"])
            response_text = str(response)
        except Exception as e:
            response_text = f"ERROR: {str(e)}"
        
        results.append({
            "input": test_case["input"],
            "category": test_case["category"],
            "expected_label": test_case["expected_label"],
            "evaluator": test_case["evaluator"],
            "response": response_text[:200] + "..." if len(response_text) > 200 else response_text
        })
    
    results_df = pd.DataFrame(results)
    return results_df


print("Regression suite runner defined.")
print("Run with: results = run_regression_suite(agent, regression_dataset, ax_client)")

In [ ]:
# Run the regression suite
print("Running regression test suite...")
print("=" * 60)

regression_results = run_regression_suite(agent, regression_dataset, ax_client)

print("\n" + "=" * 60)
print("Regression suite complete. Traces sent to Arize AX.")
print("Run evaluators on these traces to compare against expected labels.")

In [ ]:
# Run evaluations on regression traces
regression_eval_results = ax_client.run_evaluations(
    project_name=ARIZE_PROJECT_NAME,
    evaluators=["graceful_handling", "response_relevance", "tool_usage_correctness"]
)

print("Regression evaluations complete.")
print("Check Arize AX dashboard for detailed results per span.")

### Prompt Iteration Example

If the graceful_handling evaluator flags failures, we refine the system prompt and re-run the regression suite to verify the fix without breaking other behaviors.

In [ ]:
# Example: Refined system prompt after evaluation feedback
# The key change is making the graceful handling instruction more explicit

REFINED_SYSTEM_PROMPT = """
You are a helpful restaurant assistant agent. You help customers with:
- Finding restaurants and their details (menus, hours, locations, policies)
- Making new reservations
- Looking up existing booking details
- Cancelling reservations

CRITICAL BEHAVIORAL RULES:

1. ALWAYS use the retrieve tool to search for restaurant information first.

2. IMPOSSIBLE/UNREALISTIC REQUESTS: If a user asks for something that is physically 
   impossible or clearly unrealistic (examples: restaurant on the moon, restaurant 
   in Atlantis, booking for 1000 people), you MUST:
   a) Clearly state that the request cannot be fulfilled and explain why
   b) Do NOT imply you can search for or find impossible things
   c) Suggest realistic alternatives (e.g., space-themed restaurants, nearby 
      locations, splitting into multiple bookings)

3. Be polite, concise, and accurate.
4. Confirm booking details before creating a reservation.
5. If you don't have information about a real request, say so honestly.

Format your responses using <answer> tags.
"""

# Rebuild agent with refined prompt
refined_agent = Agent(
    model=bedrock_model,
    system_prompt=REFINED_SYSTEM_PROMPT,
    tools=[create_booking, get_booking_details, delete_booking, current_time, retrieve]
)

print("Refined agent created with improved graceful handling instructions.")
print("Re-run regression suite to validate improvements.")

In [ ]:
# Re-run regression suite with refined agent
print("Re-running regression suite with refined prompt...")
print("=" * 60)

refined_results = run_regression_suite(refined_agent, regression_dataset, ax_client)

print("\n" + "=" * 60)
print("Refined regression suite complete.")
print("Compare evaluation scores between original and refined prompts in Arize AX.")

## 8. Production Monitoring Setup

For production deployment, we configure:
- **Dashboards** for real-time visibility into agent performance
- **Alerts** for quality degradation, latency spikes, and cost anomalies
- **Continuous evaluation** running on sampled production traces

In [ ]:
# Define monitoring configuration
monitoring_config = {
    "project": ARIZE_PROJECT_NAME,
    
    # Dashboard panels
    "dashboard": {
        "name": "Restaurant Agent - Production Monitor",
        "panels": [
            {
                "title": "Request Volume",
                "type": "time_series",
                "metric": "trace_count",
                "granularity": "1h"
            },
            {
                "title": "Average Latency (p50/p95/p99)",
                "type": "time_series",
                "metric": "latency_percentiles",
                "percentiles": [50, 95, 99]
            },
            {
                "title": "Token Usage & Estimated Cost",
                "type": "time_series",
                "metrics": ["total_tokens", "estimated_cost_usd"],
                "granularity": "1h"
            },
            {
                "title": "Evaluation Scores Over Time",
                "type": "time_series",
                "metrics": [
                    "graceful_handling_avg_score",
                    "response_relevance_avg_score",
                    "tool_usage_correctness_avg_score"
                ]
            },
            {
                "title": "Error Rate",
                "type": "time_series",
                "metric": "error_rate",
                "granularity": "1h"
            },
            {
                "title": "Tool Usage Distribution",
                "type": "pie_chart",
                "metric": "tool_call_distribution"
            }
        ]
    },
    
    # Alert rules
    "alerts": [
        {
            "name": "High Latency Alert",
            "condition": "p95_latency > 10s",
            "window": "15m",
            "severity": "warning",
            "notification": "slack"
        },
        {
            "name": "Graceful Handling Degradation",
            "condition": "graceful_handling_avg_score < 0.8",
            "window": "1h",
            "severity": "critical",
            "notification": "slack+email"
        },
        {
            "name": "Relevance Score Drop",
            "condition": "response_relevance_avg_score < 0.7",
            "window": "1h",
            "severity": "warning",
            "notification": "slack"
        },
        {
            "name": "Cost Spike Alert",
            "condition": "hourly_cost > $50",
            "window": "1h",
            "severity": "warning",
            "notification": "email"
        },
        {
            "name": "Error Rate Spike",
            "condition": "error_rate > 5%",
            "window": "15m",
            "severity": "critical",
            "notification": "slack+email+pagerduty"
        }
    ]
}

print("Monitoring Configuration")
print("=" * 50)
print(f"Dashboard: {monitoring_config['dashboard']['name']}")
print(f"Panels: {len(monitoring_config['dashboard']['panels'])}")
print(f"Alert rules: {len(monitoring_config['alerts'])}")
print()
print("Alert Rules:")
for alert in monitoring_config['alerts']:
    print(f"  [{alert['severity'].upper()}] {alert['name']}: {alert['condition']}")

In [ ]:
# Apply monitoring configuration to Arize AX
# (In practice, dashboards and alerts are configured through the Arize UI
# or via the Arize API. Below is the API-based approach.)

# Create dashboard
dashboard = ax_client.create_dashboard(
    project_name=ARIZE_PROJECT_NAME,
    name=monitoring_config["dashboard"]["name"],
    panels=monitoring_config["dashboard"]["panels"]
)
print(f"Dashboard created: {monitoring_config['dashboard']['name']}")

# Create alert rules
for alert_config in monitoring_config["alerts"]:
    alert = ax_client.create_alert(
        project_name=ARIZE_PROJECT_NAME,
        name=alert_config["name"],
        condition=alert_config["condition"],
        window=alert_config["window"],
        severity=alert_config["severity"],
        notification_channel=alert_config["notification"]
    )
    print(f"Alert created: {alert_config['name']}")

print("\nProduction monitoring fully configured.")

In [ ]:
# Configure continuous evaluation on production traces
# This samples a percentage of production traffic for automated evaluation

continuous_eval_config = ax_client.configure_continuous_evaluation(
    project_name=ARIZE_PROJECT_NAME,
    evaluators=["graceful_handling", "response_relevance", "tool_usage_correctness"],
    sample_rate=0.1,  # Evaluate 10% of production traces
    schedule="every_hour"
)

print("Continuous evaluation configured.")
print("- Sample rate: 10% of production traces")
print("- Schedule: Every hour")
print("- Evaluators: graceful_handling, response_relevance, tool_usage_correctness")

## 9. Summary & Key Takeaways

### What we built here:

**Instrumentation:**
- OpenTelemetry tracing integrated with Strands SDK
- Traces exported to Arize AX with full span hierarchy
- Trace Tree, Agent Graph, and Timeline views available

**Evaluation:**
- Three LLM-as-a-Judge evaluators: graceful_handling, response_relevance, tool_usage_correctness
- Automated evaluation pipeline running against traced data
- Regression dataset with 7 test cases across 4 categories

**Prompt Optimization:**
- Identified graceful_handling failure (score 0) via automated eval
- Refined system prompt to explicitly handle impossible requests
- Validated fix against the regression suite without breaking other behaviors

**Production Monitoring:**
- Dashboard with 6 panels (volume, latency, cost, eval scores, errors, tool distribution)
- 5 alert rules covering latency, quality, cost, and errors
- Continuous evaluation on 10% of production traffic

---

### The core lesson:
Nondeterministic systems fail in ways you cannot predict by reading code. Without tracing and automated evaluations, failures ship silently. Observability is not optional infrastructure for agents -- it is the infrastructure.